# Buenos Aires Cultural Events Dataset

### BUSINESS CHALLENGE:

Create a product that allows us to see a quick view of current cultural events in Buenos Aires classified by type of event and venue. It should provide information about all events and related links from different venues in Buenos Aires, so we can easily pick what we want to do this week. 

In [1]:
import os
import pandas as pd
print("cwd:", os.getcwd())
print("src exists:", os.path.exists(os.path.abspath("../src")))
print("scraper exists:", os.path.exists(os.path.abspath("../src/scraper.py")))

cwd: /Users/victoriayuzova/Data-Science-Projects/ba-events-recommender/notebooks
src exists: True
scraper exists: True


In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
import os
import sys
import json
import importlib

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Force the notebook to import the *local* ../src/scraper.py (not the pip package named `scraper`)
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
if "scraper" in sys.modules:
    del sys.modules["scraper"]

scraper = importlib.import_module("scraper")
fetch_website_links = scraper.fetch_website_links
fetch_website_contents = scraper.fetch_website_contents
extract_links_from_homepages = scraper.extract_links_from_homepages
build_classified_events_dataset = scraper.build_classified_events_dataset

/Users/victoriayuzova/Data-Science-Projects/ba-events-recommender/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
import json
from pathlib import Path

p = Path.cwd().parent / "src" / "homepage_urls.json"  # because cwd is notebooks/
with p.open("r", encoding="utf-8") as f:
    homepage_urls = json.load(f)["homepage_urls"]

homepage_urls

['https://complejoteatral.gob.ar/',
 'https://malba.org.ar/',
 'https://turismo.buenosaires.gob.ar/es/eventos',
 'https://www.bellasartes.gob.ar/agenda/']

In [4]:
df_links = extract_links_from_homepages(
    homepage_urls=homepage_urls,
    fetch_website_links_fn=fetch_website_links,
    keep_same_domain=False,  # important for ticket domains
    out_dir="../data/raw",   # relative path is cleaner
    filename="01_all_links.csv"
)

df_links.head()

Saved 399 rows to: ../data/raw/2026-03-16/01_all_links.csv


,run_date,scraped_at,page_url,event_url_raw,event_url_abs,link_id
0,2026-03-16,2026-03-16T11:37:21,https://complejoteatral.gob.ar/,http://buenosaires.gob.ar/,http://buenosaires.gob.ar/,3cd0a6fbe45556781bd85be6f1ab1d74
1,2026-03-16,2026-03-16T11:37:21,https://complejoteatral.gob.ar/,https://complejoteatral.gob.ar/agenda?fecha=16...,https://complejoteatral.gob.ar/agenda?fecha=16...,e8293ea7a4942581ca1d7be79c4f5e9c
2,2026-03-16,2026-03-16T11:37:21,https://complejoteatral.gob.ar/,https://complejoteatral.gob.ar/pdf/temporada20...,https://complejoteatral.gob.ar/pdf/temporada20...,bb6f5d43ebd6fb57389f315068d57fee
3,2026-03-16,2026-03-16T11:37:21,https://complejoteatral.gob.ar/,https://complejoteatral.gob.ar/ver/visitas_gui...,https://complejoteatral.gob.ar/ver/visitas_gui...,490e1ba143103982811544dfc0542a9b
4,2026-03-16,2026-03-16T11:37:21,https://complejoteatral.gob.ar/,https://complejoteatral.gob.ar/paginas/teatros...,https://complejoteatral.gob.ar/paginas/teatros...,8393f13dbdf20717b5095e801cf12ac3


## Step 1. Use LLM to pick relevant links

### We will call an LLM so it picks only relevant links with events

We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding - hard coding each scenario would take us quite some time.

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Default model you can use elsewhere
MODEL = 'gpt-5-nano'

# Use a steadier model for link selection (avoid long hangs/timeouts)
LINK_MODEL = 'gpt-4.1-mini'

# Add a client-side timeout so a single slow request doesn't hang the notebook
openai = OpenAI(timeout=180)

In [6]:
# 1) Group candidates per homepage (IMPORTANT: use the filtered df2)
grouped = df_links.groupby("page_url")["event_url_raw"].apply(list).to_dict()

payloads = [
    {"homepage_url": page_url, "institution": None, "links": links}
    for page_url, links in grouped.items()
]

link_system_prompt = """
You are selecting event-related links for a Buenos Aires cultural events scraper.

Input JSON keys:
- homepage_url: string
- institution: string or null
- links: array of absolute URLs (strings)

Pick ONLY links useful to find current/upcoming cultural events:
- event listing pages (agenda / programacion / calendario)
- event detail pages
- ticket purchase pages
- (optional) program PDFs (ONLY if they are clearly about programming/season/events)

Exclude:
- terms/privacy, about, contact, donations, newsletters, login, accessibility pages, generic navigation
- social media

Rules:
- Do NOT invent URLs. Every URL must be exactly one of input.links
- Return at most 30 links
- Return ONLY valid JSON (no markdown/no prose) with schema:

{
  "homepage_url": "<string>",
  "institution": null,
  "links": [{"url": "<string>"}]
}
""".strip()

def select_relevant_links(payload: dict) -> dict:
    # Make sure content is string (fixes your earlier BadRequestError)
    payload_text = json.dumps(payload, ensure_ascii=False)

    print(f"Selecting relevant links for {payload['homepage_url']} by calling {LINK_MODEL}")

    response = openai.chat.completions.create(
        model=LINK_MODEL,  # use your steadier model here
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": payload_text},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )

    result = json.loads(response.choices[0].message.content)

    # Safety: keep only URLs that were in the input (no hallucinations)
    input_set = set(payload["links"])
    cleaned = [x for x in result.get("links", []) if x.get("url") in input_set]

    result["homepage_url"] = payload["homepage_url"]
    result["institution"] = payload.get("institution")
    result["links"] = cleaned[:30]

    print(f"Found {len(result['links'])} relevant links")
    return result

outs = [select_relevant_links(p) for p in payloads]

df_selected = pd.concat(
    [
        pd.DataFrame(o["links"]).assign(
            homepage_url=o.get("homepage_url"),
            institution=o.get("institution")
        )
        for o in outs
    ],
    ignore_index=True
)

df_selected.head()

Selecting relevant links for https://complejoteatral.gob.ar/ by calling gpt-4.1-mini
Found 30 relevant links
Selecting relevant links for https://malba.org.ar/ by calling gpt-4.1-mini
Found 22 relevant links
Selecting relevant links for https://turismo.buenosaires.gob.ar/es/eventos by calling gpt-4.1-mini
Found 30 relevant links
Selecting relevant links for https://www.bellasartes.gob.ar/agenda/ by calling gpt-4.1-mini
Found 14 relevant links


,url,homepage_url,institution
0,https://complejoteatral.gob.ar/agenda?fecha=16...,https://complejoteatral.gob.ar/,None
1,https://complejoteatral.gob.ar/pdf/temporada20...,https://complejoteatral.gob.ar/,None
2,https://complejoteatral.gob.ar/ver/visitas_gui...,https://complejoteatral.gob.ar/,None
3,https://complejoteatral.gob.ar/ver/GERALD-CLAY...,https://complejoteatral.gob.ar/,None
4,https://www.tuentrada.com/jazz-bsas-ctba,https://complejoteatral.gob.ar/,None


In [7]:
df_selected.groupby("homepage_url")["url"].count().sort_values(ascending=False)

homepage_url
https://complejoteatral.gob.ar/                  30
https://turismo.buenosaires.gob.ar/es/eventos    30
https://malba.org.ar/                            22
https://www.bellasartes.gob.ar/agenda/           14
Name: url, dtype: int64

In [8]:
from datetime import datetime
import os

run_date = datetime.utcnow().strftime("%Y-%m-%d")

save_folder = f"../data/raw/{run_date}"
os.makedirs(save_folder, exist_ok=True)

df_selected.to_csv(f"{save_folder}/02_relevant_links.csv", index=False)

In [9]:
link_system_prompt = """
You are selecting event-related links for a Buenos Aires cultural events scraper.

You will receive a JSON object as input with keys:
- homepage_url: string
- institution: string or null
- links: array of absolute URLs (strings)

Your job:
- Pick ONLY links that are relevant for finding current/upcoming cultural events (agenda/listings, event detail pages, ticket purchase pages, program PDFs, calendars).
- Exclude terms/privacy, contact/about, donations/sponsors, newsletters, login, generic navigation, accessibility pages, and any social media.
- Do NOT invent new URLs. Every returned URL must come from input.links.
- Return at most 30 links.

Return ONLY valid JSON (no markdown, no prose) with this schema:

{
  "homepage_url": "<string>",
  "institution": "<string or null>",
  "links": [
    {
      "url": "<string>"
    }
  ]
}
"""

In [10]:
def get_links_user_prompt(payload_json_links):
    user_prompt = f"""
Here is the list of links on the website in json format: {payload_json_links} -
Please decide which of these are relevant web links for a brochure listing current cultural
evens in Buenos Aires.
Do not include Terms of Service, Privacy, email, social media links, or general descriptions of the theater that´s not related to any event.

Links (some might be relative links):

"""
    links = fetch_website_links(payload_json_links)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
def select_relevant_links(payload_json_links):
    print(f"Selecting relevant links for {payload_json_links} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": payload_json_links}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links
    

In [12]:
for i, p in enumerate(payloads):
    if not isinstance(p, str):
        print(i, type(p))
        break


0 <class 'dict'>


In [13]:
outs = [select_relevant_links(json.dumps(p, ensure_ascii=False)) for p in payloads]

df_selected = pd.concat(
    [pd.DataFrame(o["links"]).assign(homepage_url=o.get("homepage_url"), institution=o.get("institution"))
     for o in outs],
    ignore_index=True
)

df_selected

Selecting relevant links for {"homepage_url": "https://complejoteatral.gob.ar/", "institution": null, "links": ["http://buenosaires.gob.ar/", "https://complejoteatral.gob.ar/agenda?fecha=16-03-2026", "https://complejoteatral.gob.ar/pdf/temporada2026.pdf", "https://complejoteatral.gob.ar/ver/visitas_guiadas_al_teatro_san_martín", "https://complejoteatral.gob.ar/paginas/teatros-accesibles", "https://complejoteatral.gob.ar/paginas/info-contacto", "https://www.instagram.com/elSanMartinCTBA", "https://www.twitter.com/elSanMartinCTBA", "https://www.facebook.com/TeatroSanMartinCTBA", "https://complejoteatral.gob.ar/#programacion", "/noticias/", "https://complejoteatral.gob.ar/companias-estables/ballet-contemporaneo", "https://complejoteatral.gob.ar/companias-estables/grupo-de-titiriteros", "https://complejoteatral.gob.ar/paginas/teatros", "https://complejoteatral.gob.ar/paginas/legado-y-mision", "https://complejoteatral.gob.ar/paginas/autoridades", "https://complejoteatral.gob.ar/paginas/espa

,url,homepage_url,institution
0,https://complejoteatral.gob.ar/agenda?fecha=16...,https://complejoteatral.gob.ar/,None
1,https://complejoteatral.gob.ar/pdf/temporada20...,https://complejoteatral.gob.ar/,None
2,https://complejoteatral.gob.ar/ver/GERALD-CLAY...,https://complejoteatral.gob.ar/,None
3,https://complejoteatral.gob.ar/ver/la_gaviota,https://complejoteatral.gob.ar/,None
4,https://complejoteatral.gob.ar/ver/jazz_buenos...,https://complejoteatral.gob.ar/,None
...,...,...,...
98,/agenda/entre-tela/,https://www.bellasartes.gob.ar/agenda/,None
99,/agenda/catalogo-dibujos-antiguos/,https://www.bellasartes.gob.ar/agenda/,None
100,/agenda/visita-acc_1/,https://www.bellasartes.gob.ar/agenda/,None
101,/lang/pt/?path=/agenda/,https://www.bellasartes.gob.ar/agenda/,None


In [14]:
df_selected.to_csv("relevant_links.csv", index=False)



## Second step: lets classify those links

Assemble all the details into another prompt to GPT-5-nano

In [15]:

df_events = build_classified_events_dataset(
    df_relevant = pd.read_csv("relevant_links.csv"),  # df_selected already has url + homepage_url
    fetch_website_contents_fn=fetch_website_contents,
    openai_client=openai,
    model="gpt-4.1-mini",
    out_dir="../data/raw",
    filename="03_events.csv",
    #limit=30,
)
df_events.head()

Saved 103 rows to: ../data/raw/2026-03-16/03_events.csv


,url,homepage_url,page_type,title,summary,category,start_date,start_time,venue,price,is_free,tags,confidence,run_date,extracted_at
0,https://complejoteatral.gob.ar/agenda?fecha=16...,https://complejoteatral.gob.ar/,listing,Agenda | Complejo Teatral de Buenos Aires,Listing of theatrical events around 16 March 2...,theatre,None,None,None,None,None,"[teatro, agenda, temporada internacional, comp...",0.80,2026-03-16,2026-03-16T11:43:12+00:00
1,https://complejoteatral.gob.ar/pdf/temporada20...,https://complejoteatral.gob.ar/,pdf,None,None,other,None,None,None,None,None,[],0.30,2026-03-16,2026-03-16T11:43:12+00:00
2,https://complejoteatral.gob.ar/ver/GERALD-CLAY...,https://complejoteatral.gob.ar/,event_detail,GERALD CLAYTON TRIO,Comienza el ciclo de jazz en el Teatro San Mar...,music,2026-03-17,20:00,"Teatro San Martín, Sala Martín Coronado","Platea Filas 2 a 5 $85.000, Platea Filas 6 a 2...",False,"[jazz, Gerald Clayton, Trio, Teatro San Martín...",0.95,2026-03-16,2026-03-16T11:43:12+00:00
3,https://complejoteatral.gob.ar/ver/la_gaviota,https://complejoteatral.gob.ar/,event_detail,La Gaviota,"Obra de teatro de Antón Chéjov, dirigida por R...",theatre,None,None,"Teatro San Martín, Sala Casacuberta","Platea $21.000, Miércoles $12.000",False,"[teatro, obra, Antón Chéjov, Rubén Szuchmacher...",0.95,2026-03-16,2026-03-16T11:43:12+00:00
4,https://complejoteatral.gob.ar/ver/jazz_buenos...,https://complejoteatral.gob.ar/,event_detail,Jazz Buenos Aires,Ciclo de nueve conciertos de Jazz at Lincoln C...,music,2026-03-17,20:00,"Teatro San Martín, Sala Martín Coronado","Platea Filas 2 a 5 $85.000, Platea Filas 6 a 2...",False,"[jazz, concierto, música, Jazz at Lincoln Cent...",0.95,2026-03-16,2026-03-16T11:43:12+00:00


In [20]:
df_events.shape

(103, 15)

In [21]:
df_events.groupby("homepage_url")["homepage_url"].count().sort_values(ascending=False)

homepage_url
https://malba.org.ar/                            30
https://turismo.buenosaires.gob.ar/es/eventos    29
https://complejoteatral.gob.ar/                  28
https://www.bellasartes.gob.ar/agenda/           16
Name: homepage_url, dtype: int64

In [22]:
df_events.groupby("category")["homepage_url"].count().sort_values(ascending=False)

category
other         61
theatre       10
cinema         9
exhibition     8
talk           8
workshop       4
music          2
dance          1
Name: homepage_url, dtype: int64

In [19]:
df_events

,url,homepage_url,page_type,title,summary,category,start_date,start_time,venue,price,is_free,tags,confidence,run_date,extracted_at
0,https://complejoteatral.gob.ar/agenda?fecha=16...,https://complejoteatral.gob.ar/,listing,Agenda | Complejo Teatral de Buenos Aires,Listing of theatrical events around 16 March 2...,theatre,None,None,None,None,None,"[teatro, agenda, temporada internacional, comp...",0.80,2026-03-16,2026-03-16T11:43:12+00:00
1,https://complejoteatral.gob.ar/pdf/temporada20...,https://complejoteatral.gob.ar/,pdf,None,None,other,None,None,None,None,None,[],0.30,2026-03-16,2026-03-16T11:43:12+00:00
2,https://complejoteatral.gob.ar/ver/GERALD-CLAY...,https://complejoteatral.gob.ar/,event_detail,GERALD CLAYTON TRIO,Comienza el ciclo de jazz en el Teatro San Mar...,music,2026-03-17,20:00,"Teatro San Martín, Sala Martín Coronado","Platea Filas 2 a 5 $85.000, Platea Filas 6 a 2...",False,"[jazz, Gerald Clayton, Trio, Teatro San Martín...",0.95,2026-03-16,2026-03-16T11:43:12+00:00
3,https://complejoteatral.gob.ar/ver/la_gaviota,https://complejoteatral.gob.ar/,event_detail,La Gaviota,"Obra de teatro de Antón Chéjov, dirigida por R...",theatre,None,None,"Teatro San Martín, Sala Casacuberta","Platea $21.000, Miércoles $12.000",False,"[teatro, obra, Antón Chéjov, Rubén Szuchmacher...",0.95,2026-03-16,2026-03-16T11:43:12+00:00
4,https://complejoteatral.gob.ar/ver/jazz_buenos...,https://complejoteatral.gob.ar/,event_detail,Jazz Buenos Aires,Ciclo de nueve conciertos de Jazz at Lincoln C...,music,2026-03-17,20:00,"Teatro San Martín, Sala Martín Coronado","Platea Filas 2 a 5 $85.000, Platea Filas 6 a 2...",False,"[jazz, concierto, música, Jazz at Lincoln Cent...",0.95,2026-03-16,2026-03-16T11:43:12+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,/agenda/entre-tela/,https://www.bellasartes.gob.ar/agenda/,other,None,None,other,None,None,None,None,None,[],0.10,2026-03-16,2026-03-16T11:43:12+00:00
99,/agenda/catalogo-dibujos-antiguos/,https://www.bellasartes.gob.ar/agenda/,other,None,None,other,None,None,None,None,None,[],0.10,2026-03-16,2026-03-16T11:43:12+00:00
100,/agenda/visita-acc_1/,https://www.bellasartes.gob.ar/agenda/,event_detail,None,None,other,None,None,None,None,None,[],0.10,2026-03-16,2026-03-16T11:43:12+00:00
101,/lang/pt/?path=/agenda/,https://www.bellasartes.gob.ar/agenda/,listing,None,None,other,None,None,None,None,None,[],0.20,2026-03-16,2026-03-16T11:43:12+00:00


In [25]:
pd.set_option('display.max_colwidth', 80)

df_to_rate = df_events[df_events['page_type'] == 'event_detail'][['url', 'title', 'category', 'is_free', 'summary']].reset_index(drop=True)
df_to_rate

,url,title,category,is_free,summary
0,https://complejoteatral.gob.ar/ver/GERALD-CLAYTON-TRIO,GERALD CLAYTON TRIO,music,False,Comienza el ciclo de jazz en el Teatro San Martín con la presentación de Ger...
1,https://complejoteatral.gob.ar/ver/la_gaviota,La Gaviota,theatre,False,"Obra de teatro de Antón Chéjov, dirigida por Rubén Szuchmacher, con traducci..."
2,https://complejoteatral.gob.ar/ver/jazz_buenos_aires,Jazz Buenos Aires,music,False,"Ciclo de nueve conciertos de Jazz at Lincoln Center en el Teatro San Martín,..."
3,https://complejoteatral.gob.ar/ver/los-pilares-de-la-sociedad,LOS PILARES DE LA SOCIEDAD,theatre,False,Obra de teatro de Henrik Ibsen que denuncia la hipocresía moral y social de ...
4,https://complejoteatral.gob.ar/ver/invasiones-1,INVASIONES I,theatre,False,Invasiones I: No Bombardeen Buenos Aires es un musical protagonizado por Ele...
5,https://complejoteatral.gob.ar/ver/paraiso,PARAÍSO,theatre,False,Paraíso es una obra que interpela la masculinidad. Es la historia de Juan Va...
6,https://complejoteatral.gob.ar/ver/baco-polaco,Baco polaco,theatre,False,Obra teatral de Mauricio Kartun que traslada la tragedia Las bacantes de Eur...
7,https://complejoteatral.gob.ar/ver/festival_el_tornillo,FESTIVAL EL TORNILLO - II EDICIÓN,theatre,None,Festival El Tornillo es un encuentro artístico creado y gestionado por los t...
8,https://complejoteatral.gob.ar/ver/centroamerica,CENTROAMÉRICA,theatre,False,Obra teatral del colectivo mexicano Lagartijas Tiradas al Sol que examina la...
9,https://complejoteatral.gob.ar/ver/numancia,NUMANCIA,theatre,False,"Obra teatral dirigida por José Luis Alonso de Santos, inspirada en hechos hi..."


In [30]:

liked = [0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0]

df_to_rate['liked'] = liked
df_to_rate.to_csv("../data/processed/user_ratings.csv", index=False)